In [ ]:
from pathlib import Path

# Run this baseline for the attached CodeCodeXGLUE V3 V3 Kaggle dataset.
# Add the CodeXGLUE V3 dataset to the Kaggle notebook inputs before running all cells.
DATASET_KEYS = ("codexglue",)
RUN_LABEL = "snn_baselines"


# Kaggle's current PyTorch build cannot execute kernels on Tesla P100 (sm_60).
# These notebooks are intended for a T4-class accelerator; two T4s are fine,
# although this single-process implementation uses GPU 0.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
_GPU_CAPABILITY = torch.cuda.get_device_capability(0)
_GPU_NAME = torch.cuda.get_device_name(0)
if _GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{_GPU_NAME} has unsupported CUDA capability sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}. "
        "Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
    )
print({"gpu": _GPU_NAME, "capability": f"sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}", "gpu_count": torch.cuda.device_count()})
# Runtime profile. Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "final_full"
RUN_PRESETS = {"quick_1h": {'max_train_pairs': 50000, 'max_valid_pairs': 10000, 'max_test_pairs': 10000, 'epochs': 10, 'patience': 3}, "extended_6_7h": {'max_train_pairs': 100000, 'max_valid_pairs': 20000, 'max_test_pairs': 20000, 'epochs': 50, 'patience': 8}}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
}

# Final paper protocol: use every available pair in each official split.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
}

# --- bounded run budget (scripts/patch_kaggle_run_budget.py) ---
# Kaggle sessions are capped, and a run that dies at the limit produces nothing.
# Training data stays large so results remain comparable with the published
# table; validation and test are capped because a bigger validation split only
# sharpens one threshold, and a bigger test split only tightens an error bar we
# do not report.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = RUN_PRESETS[RUN_PROFILE]


In [ ]:
# === per-language breakdown helper (patched by scripts/patch_kaggle_language_breakdown.py) ===
# Splits an already-computed set of test predictions by the language of each
# pair. No retraining and no separate per-language model: this is the same run,
# reported per language so a strong average cannot hide a collapsed language.
import gzip as _gzip
import json as _json
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

_LANGUAGE_CACHE = {}
LANGUAGE_BREAKDOWN_ROWS = []


def _resolve_codes_file():
    for root in (_Path("/kaggle/input"), _Path("/kaggle/working"), _Path(".")):
        if not root.exists():
            continue
        for name in ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"):
            for path in root.rglob(name):
                if path.is_file():
                    return path
    return None


def _open_any(path):
    with open(path, "rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return _gzip.open(path, "rt", encoding="utf-8") if packed else open(path, "r", encoding="utf-8")


def code_languages():
    """``code_id -> language`` from the attached clean-data bundle."""
    if _LANGUAGE_CACHE:
        return _LANGUAGE_CACHE
    path = _resolve_codes_file()
    if path is None:
        print("[language-breakdown] codes.jsonl not found; breakdown will be skipped.")
        return _LANGUAGE_CACHE
    with _open_any(path) as stream:
        for line in stream:
            if not line.strip():
                continue
            record = _json.loads(line)
            code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
            _LANGUAGE_CACHE[code_id] = str(record.get("language", record.get("lang", "unknown")))
    print(f"[language-breakdown] languages loaded for {len(_LANGUAGE_CACHE):,} codes.")
    return _LANGUAGE_CACHE


def _binary_scores(labels, predicted):
    labels = _np.asarray(labels, dtype=_np.int64)
    predicted = _np.asarray(predicted, dtype=_np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "P": precision, "R": recall, "F1": f1,
        "Acc": (tp + tn) / max(1, len(labels)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Pairs": int(len(labels)), "Positives": int((labels == 1).sum()),
    }


def record_language_breakdown(frame, scores, threshold, *, dataset, method, graph_type=None):
    """Partition this run's test predictions by pair language and record them."""
    languages = code_languages()
    if not languages or frame is None or not len(frame):
        return []
    scores = _np.asarray(scores, dtype=_np.float64).reshape(-1)
    labels = _np.asarray(frame["label"], dtype=_np.int64).reshape(-1)
    if len(scores) != len(labels):
        print(f"[language-breakdown] skipped {method}: {len(scores)} scores vs {len(labels)} labels.")
        return []
    predicted = (scores >= float(threshold)).astype(_np.int64)

    left = [languages.get(str(value), "unknown") for value in frame["left_id"]]
    right = [languages.get(str(value), "unknown") for value in frame["right_id"]]
    # Cross-language pairs get their own bucket instead of being attributed to
    # one side; ATCoder is entirely java<->python and would otherwise vanish.
    keys = [a if a == b else f"{min(a, b)}->{max(a, b)}" for a, b in zip(left, right)]

    rows = []
    for key in sorted(set(keys)):
        mask = _np.asarray([value == key for value in keys])
        row = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": key}
        row.update(_binary_scores(labels[mask], predicted[mask]))
        row["Threshold"] = float(threshold)
        rows.append(row)
    overall = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": "ALL"}
    overall.update(_binary_scores(labels, predicted))
    overall["Threshold"] = float(threshold)
    rows.append(overall)

    LANGUAGE_BREAKDOWN_ROWS.extend(rows)
    table = _pd.DataFrame(LANGUAGE_BREAKDOWN_ROWS)
    out_path = _Path("/kaggle/working") / f"{dataset}_language_breakdown.csv"
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
    except OSError:
        out_path = _Path(f"{dataset}_language_breakdown.csv")
        table.to_csv(out_path, index=False)
    print(f"\n[language-breakdown] {method}{'/' + graph_type if graph_type else ''}")
    print(_pd.DataFrame(rows)[["Language", "P", "R", "F1", "Acc", "Pairs", "Positives"]].to_string(index=False))
    print(f"[language-breakdown] written to {out_path}")
    return rows

DATASET_KEY_FOR_BREAKDOWN = "codexglue_v3"


# CodeXGLUE V3 SNN Baselines

Kaggle-ready Siamese Neural Network baseline for CodeXGLUE V3 spectral representations.
It trains one shared-encoder pair classifier for each graph type: AST, CFG, PDG, DDG, and CPG.


In [ ]:
# Executes the unchanged baseline pipeline once per dataset in isolated state.
# This run writes CodeCodeXGLUE V3 V3-only CSV files, for example xglue4_*_results.csv.
def run_one_dataset(dataset_key: str):
    # =========================
    # Config
    # =========================
    from pathlib import Path
    import time

    # End-to-end method runtime: loading + preprocessing + training + evaluation.
    run_started = time.perf_counter()

    DATASET_KEY = dataset_key
    KAGGLE_DATA_ROOT = Path("/kaggle/input") / DATASET_KEY
    WORK_DIR = Path("/kaggle/working")

    # ``"auto"`` discovers every graph layer serialized in graph_spectra
    # (currently AST, CFG, DDG and CPG).  Supply a list such as ["ast", "cfg"]
    # only when intentionally running a selected subset.
    GRAPH_TYPES = "auto"

    # Fixed-size spectral input for the MLP encoder.
    # Keep 64 for comparability with the local RF/LR baselines; try 128 or 300 for a heavier run.
    K_EIGEN = 64
    USE_EIGEN_STATS = True
    USE_GRAPH_STATS = False

    # Full dataset run by default.
    MAX_TRAIN_PAIRS = RUN_CONFIG["max_train_pairs"]
    MAX_VALID_PAIRS = RUN_CONFIG["max_valid_pairs"]
    MAX_TEST_PAIRS = RUN_CONFIG["max_test_pairs"]

    EPOCHS = RUN_CONFIG["epochs"]
    BATCH_SIZE = 8192
    HIDDEN_DIM = 256
    EMBED_DIM = 128
    DROPOUT = 0.10
    LEARNING_RATE = 1e-3
    WEIGHT_DECAY = 1e-4
    PATIENCE = RUN_CONFIG["patience"]
    USE_AMP = True
    USE_POS_WEIGHT = False
    # Kaggle/Jupyter can print harmless DataLoader multiprocessing cleanup errors.
    # Keep this at 0 unless you specifically need multiprocessing workers.
    NUM_WORKERS = 0
    SEED = 42

    DEVICE = "cuda"

    RESULTS_PATH = WORK_DIR / f"{DATASET_KEY}_snn_baseline_results.csv"
    HISTORY_PATH = WORK_DIR / f"{DATASET_KEY}_snn_training_history.csv"
    PLOT_PATH = WORK_DIR / f"{DATASET_KEY}_snn_training_curves.png"

    # =========================
    # Imports and file helpers
    # =========================
    import gc
    import gzip
    import json
    import math
    import os
    import random
    import zipfile
    from contextlib import nullcontext
    from dataclasses import dataclass

    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset
    from tqdm.auto import tqdm


    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    if DEVICE == "cuda" and not torch.cuda.is_available():
        DEVICE = "cpu"
    print("Device:", DEVICE)

    if DEVICE == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True


    EXTRACT_ROOT = WORK_DIR / f"{DATASET_KEY}_clean_data_extracted"


    def _candidate_files_inside(directory: Path, names: tuple[str, ...]) -> list[Path]:
        files = []
        for name in names:
            files.extend([p for p in directory.rglob(name) if p.is_file()])
            files.extend([p for p in directory.rglob(name + ".tmp") if p.is_file()])
            files.extend([p for p in directory.rglob(name + ".gz.tmp") if p.is_file()])
        return files


    def _candidate_roots() -> list[Path]:
        roots = [
            KAGGLE_DATA_ROOT,
            KAGGLE_DATA_ROOT / "clean_data",
            EXTRACT_ROOT,
            EXTRACT_ROOT / "clean_data",
            Path("/kaggle/input"),
        ]
        return [root for root in roots if root.exists()]


    def _find_zip_file() -> Path | None:
        zip_names = ["Xglue.zip", "xglue.zip", "XGLUE.zip", "clean_data.zip"]
        for root in [KAGGLE_DATA_ROOT, Path("/kaggle/input")]:
            if not root.exists():
                continue
            for name in zip_names:
                direct = root / name
                if direct.is_file():
                    return direct
            zips = [p for p in root.rglob("*.zip") if p.is_file()]
            if zips:
                return sorted(zips, key=lambda p: (len(p.relative_to(root).parts), len(p.name), str(p)))[0]
        return None


    def ensure_zip_extracted() -> Path | None:
        for root in _candidate_roots():
            if _candidate_files_inside(root, ("pairs.csv.gz", "pairs.csv")):
                return None
        zip_path = _find_zip_file()
        if zip_path is None:
            return None
        marker = EXTRACT_ROOT / ".extracted_ok"
        if marker.exists():
            print("Using already extracted zip:", EXTRACT_ROOT)
            return EXTRACT_ROOT
        EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
        print("Extracting zip:", zip_path)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(EXTRACT_ROOT)
        marker.write_text(str(zip_path), encoding="utf-8")
        return EXTRACT_ROOT


    def resolve_file_path(path: Path, *fallback_names: str) -> Path:
        if path.is_file():
            return path
        if path.is_dir():
            matches = _candidate_files_inside(path, fallback_names or (path.name,))
            if matches:
                return sorted(matches, key=lambda p: (len(p.relative_to(path).parts), len(p.name), str(p)))[0]
        raise FileNotFoundError(f"Expected a file but got: {path}")


    def is_gzip_file(path: Path) -> bool:
        path = resolve_file_path(path)
        with path.open("rb") as f:
            return f.read(2) == b"\x1f\x8b"


    def open_text(path: Path):
        path = resolve_file_path(path)
        if is_gzip_file(path):
            return gzip.open(path, "rt", encoding="utf-8")
        return path.open("r", encoding="utf-8")


    def find_file(*names: str) -> Path:
        ensure_zip_extracted()
        available = []
        for root in _candidate_roots():
            for name in names:
                direct = root / name
                if direct.is_file():
                    return direct
                if direct.is_dir():
                    return resolve_file_path(direct, *names)
            matches = []
            for name in names:
                matches.extend(_candidate_files_inside(root, (name,)))
            if matches:
                return sorted(matches, key=lambda p: (len(p.relative_to(root).parts), len(p.name), str(p)))[0]
            available.extend(str(p) for p in sorted(root.rglob("*"))[:30])
        raise FileNotFoundError(
            f"Could not find any file named: {names}\n"
            f"Searched roots: {[str(r) for r in _candidate_roots()]}\n"
            f"First available paths: {available[:40]}"
        )


    pairs_path = find_file("pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    graphs_path = find_file("graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
    print("pairs:", pairs_path, "is_file=", pairs_path.is_file())
    print("graphs:", graphs_path, "is_file=", graphs_path.is_file())

    def discover_graph_types(path: Path) -> list[str]:
        """Return the union of graph layers actually stored in this attachment."""
        discovered = set()
        with open_text(path) as stream:
            for line in tqdm(stream, desc="Discovering graph layers", unit="code"):
                if not line.strip():
                    continue
                record = json.loads(line)
                graphs = record.get("graphs", {})
                if isinstance(graphs, dict):
                    discovered.update(str(name).lower() for name, layer in graphs.items() if isinstance(layer, dict))
        preferred = ["ast", "cfg", "ddg", "cpg"]
        return [name for name in preferred if name in discovered] + sorted(discovered - set(preferred))

    if isinstance(GRAPH_TYPES, str):
        if GRAPH_TYPES.lower() != "auto":
            raise ValueError('GRAPH_TYPES must be "auto" or a list of graph-layer names')
        GRAPH_TYPES = discover_graph_types(graphs_path)
    else:
        GRAPH_TYPES = [str(name).lower() for name in GRAPH_TYPES]
# --- layer support (patched by scripts/patch_kaggle_layer_support.py) ---
    UNSUPPORTED_GRAPH_LAYERS = []
    # Every language in this benchmark provides all four layers.
    GRAPH_TYPES = [layer for layer in GRAPH_TYPES if layer not in UNSUPPORTED_GRAPH_LAYERS]
    print('graph layers scored:', GRAPH_TYPES, '| dropped:', UNSUPPORTED_GRAPH_LAYERS)
    if not GRAPH_TYPES:
        raise RuntimeError("No graph layers were found in graph_spectra.")
    print("SNN graph runs:", GRAPH_TYPES)


    # =========================
    # Data and spectral vectors
    # =========================
    def load_pairs(path: Path) -> pd.DataFrame:
        path = resolve_file_path(path, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
        compression = "gzip" if is_gzip_file(path) else None
        df = pd.read_csv(path, compression=compression, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int8})
        df["left_id"] = df["left_id"].astype(str)
        df["right_id"] = df["right_id"].astype(str)
        df["label"] = df["label"].astype(np.int8)
        return df[["split", "left_id", "right_id", "label"]]


    def eigen_stats(values: list[float]) -> np.ndarray:
        arr = np.asarray(values or [], dtype=np.float32)
        if arr.size == 0:
            return np.zeros(8, dtype=np.float32)
        q25, q50, q75 = np.percentile(arr, [25, 50, 75]).astype(np.float32)
        return np.asarray([
            min(arr.size / 2000.0, 10.0),
            float(arr.mean()),
            float(arr.std()),
            float(arr.min()),
            float(arr.max()),
            float(q25),
            float(q50),
            float(q75),
        ], dtype=np.float32)


    def pad_eigen(values: list[float], k_eigen: int) -> np.ndarray:
        arr = np.asarray(values or [], dtype=np.float32)
        out = np.zeros(k_eigen, dtype=np.float32)
        if arr.size:
            take = min(k_eigen, arr.size)
            out[:take] = arr[:take]
        return out


    def graph_stats(layer: dict) -> np.ndarray:
        adjacency = layer.get("adjacency", {}) if isinstance(layer, dict) else {}
        n = int(adjacency.get("num_nodes", 0) or 0)
        raw_edges = int(adjacency.get("num_edges", 0) or 0)
        possible_edges = max(1, n * max(1, n - 1))
        density = raw_edges / possible_edges
        return np.asarray([
            np.log1p(max(n, 0)) / 10.0,
            np.log1p(max(raw_edges, 0)) / 10.0,
            min(density, 1.0),
        ], dtype=np.float32)


    def vector_from_layer(layer: dict) -> np.ndarray:
        eig = layer.get("eigenvalues", []) if isinstance(layer, dict) else []
        pieces = []
        if USE_EIGEN_STATS:
            pieces.append(eigen_stats(eig))
        pieces.append(pad_eigen(eig, K_EIGEN))
        if USE_GRAPH_STATS:
            pieces.append(graph_stats(layer))
        return np.concatenate(pieces).astype(np.float32)


    def load_code_vectors(graph_type: str) -> dict[str, np.ndarray]:
        vectors = {}
        with open_text(graphs_path) as f:
            for line in tqdm(f, desc=f"Loading {graph_type.upper()} spectra", unit="code"):
                if not line.strip():
                    continue
                row = json.loads(line)
                code_id = str(row.get("code_id"))
                layer = row.get("graphs", {}).get(graph_type, {})
                vectors[code_id] = vector_from_layer(layer)
        return vectors


    def maybe_sample(df: pd.DataFrame, max_rows: int | None, seed: int) -> pd.DataFrame:
        if max_rows is None or len(df) <= max_rows:
            return df.reset_index(drop=True)
        return df.sample(n=max_rows, random_state=seed).reset_index(drop=True)


    @dataclass
    class PairArrays:
        left_idx: np.ndarray
        right_idx: np.ndarray
        labels: np.ndarray


    def build_graph_data(pairs_df: pd.DataFrame, vectors: dict[str, np.ndarray], seed: int):
        usable = pairs_df[pairs_df.left_id.isin(vectors) & pairs_df.right_id.isin(vectors)].reset_index(drop=True)
        print("usable pairs:")
        print(usable.groupby(["split", "label"]).size())

        train_df = maybe_sample(usable[usable.split == "train"], MAX_TRAIN_PAIRS, seed)
        valid_df = maybe_sample(usable[usable.split == "valid"], MAX_VALID_PAIRS, seed + 1)
        test_df = maybe_sample(usable[usable.split == "test"], MAX_TEST_PAIRS, seed + 2)
        used_ids = sorted(set(train_df.left_id) | set(train_df.right_id) | set(valid_df.left_id) | set(valid_df.right_id) | set(test_df.left_id) | set(test_df.right_id))
        id_to_idx = {code_id: idx for idx, code_id in enumerate(used_ids)}
        matrix = np.stack([vectors[code_id] for code_id in used_ids]).astype(np.float32)

        train_ids = sorted(set(train_df.left_id) | set(train_df.right_id))
        train_rows = np.asarray([id_to_idx[code_id] for code_id in train_ids], dtype=np.int64)
        mean = matrix[train_rows].mean(axis=0, keepdims=True)
        std = matrix[train_rows].std(axis=0, keepdims=True)
        std[std < 1e-6] = 1.0
        matrix = (matrix - mean) / std

        def split_arrays(df: pd.DataFrame) -> PairArrays:
            return PairArrays(
                left_idx=df.left_id.map(id_to_idx).to_numpy(dtype=np.int64),
                right_idx=df.right_id.map(id_to_idx).to_numpy(dtype=np.int64),
                labels=df.label.to_numpy(dtype=np.float32),
            )

        # The frames are returned as well: split_arrays() reduces them to numeric
        # indices, which cannot be mapped back to code ids for the per-language
        # breakdown.
        return matrix, {
            "train": split_arrays(train_df),
            "valid": split_arrays(valid_df),
            "test": split_arrays(test_df),
        }, len(used_ids), {"train": train_df, "valid": valid_df, "test": test_df}


    pairs_df = load_pairs(pairs_path)
    print("All pairs:", len(pairs_df))
    print(pairs_df.groupby(["split", "label"]).size())


    # =========================
    # Model, metrics, training
    # =========================
    class PairIndexDataset(Dataset):
        def __init__(self, arrays: PairArrays):
            self.left_idx = torch.from_numpy(arrays.left_idx).long()
            self.right_idx = torch.from_numpy(arrays.right_idx).long()
            self.labels = torch.from_numpy(arrays.labels).float()

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, idx):
            return self.left_idx[idx], self.right_idx[idx], self.labels[idx]


    class SiameseSpectralNet(nn.Module):
        def __init__(self, input_dim: int, hidden_dim: int, embed_dim: int, dropout: float):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, embed_dim),
            )
            self.head = nn.Sequential(
                nn.Linear(embed_dim * 2 + 2, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, 1),
            )

        def forward(self, left_vecs, right_vecs):
            left = F.normalize(self.encoder(left_vecs), dim=-1)
            right = F.normalize(self.encoder(right_vecs), dim=-1)
            diff = torch.abs(left - right)
            prod = left * right
            cosine = prod.sum(dim=-1, keepdim=True)
            l2 = torch.linalg.vector_norm(left - right, dim=-1, keepdim=True)
            return self.head(torch.cat([diff, prod, cosine, l2], dim=-1)).squeeze(-1)


    def make_loader(arrays: PairArrays, shuffle: bool) -> DataLoader:
        kwargs = {
            "batch_size": BATCH_SIZE,
            "shuffle": shuffle,
            "num_workers": NUM_WORKERS,
            "pin_memory": DEVICE == "cuda",
        }
        if NUM_WORKERS > 0:
            kwargs["persistent_workers"] = True
            kwargs["prefetch_factor"] = 2
        return DataLoader(PairIndexDataset(arrays), **kwargs)


    def autocast_context(enabled: bool):
        if DEVICE == "cuda" and enabled:
            return torch.amp.autocast("cuda")
        return nullcontext()


    def make_scaler(enabled: bool):
        try:
            return torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda" and enabled))
        except TypeError:
            return torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda" and enabled))


    def metrics_at_threshold(labels: np.ndarray, scores: np.ndarray, threshold: float) -> dict:
        preds = scores >= threshold
        labels_bool = labels.astype(bool)
        tp = int(np.logical_and(preds, labels_bool).sum())
        fp = int(np.logical_and(preds, ~labels_bool).sum())
        tn = int(np.logical_and(~preds, ~labels_bool).sum())
        fn = int(np.logical_and(~preds, labels_bool).sum())
        precision = tp / max(1, tp + fp)
        recall = tp / max(1, tp + fn)
        f1 = 2 * precision * recall / max(1e-12, precision + recall)
        acc = (tp + tn) / max(1, tp + fp + tn + fn)
        return {"P": precision, "R": recall, "F1": f1, "Acc": acc, "TP": tp, "FP": fp, "TN": tn, "FN": fn, "Threshold": float(threshold)}


    def best_f1_threshold(labels: np.ndarray, scores: np.ndarray) -> float:
        candidates = np.unique(np.quantile(scores, np.linspace(0.0, 1.0, 501)))
        candidates = np.unique(np.concatenate([candidates, np.asarray([0.5], dtype=np.float32)]))
        best_thr = 0.5
        best_f1 = -1.0
        best_acc = -1.0
        for thr in candidates:
            m = metrics_at_threshold(labels, scores, float(thr))
            if (m["F1"], m["Acc"]) > (best_f1, best_acc):
                best_thr = float(thr)
                best_f1 = m["F1"]
                best_acc = m["Acc"]
        return best_thr


    @torch.no_grad()
    def predict_scores(model: nn.Module, code_tensor: torch.Tensor, loader: DataLoader, desc: str):
        model.eval()
        all_scores = []
        all_labels = []
        for left_idx, right_idx, labels in tqdm(loader, desc=desc, leave=False):
            left_idx = left_idx.to(DEVICE, non_blocking=True)
            right_idx = right_idx.to(DEVICE, non_blocking=True)
            logits = model(code_tensor[left_idx], code_tensor[right_idx])
            scores = torch.sigmoid(logits).float().cpu().numpy()
            all_scores.append(scores)
            all_labels.append(labels.numpy())
        return np.concatenate(all_labels).astype(np.float32), np.concatenate(all_scores).astype(np.float32)


    def train_one_graph(graph_type: str, pairs_df: pd.DataFrame, seed: int):
        graph_started = time.perf_counter()
        print("\n" + "=" * 80)
        print("Graph type:", graph_type.upper())
        vectors = load_code_vectors(graph_type)
        code_matrix, split_data, code_count, split_frames = build_graph_data(pairs_df, vectors, seed)
        input_dim = code_matrix.shape[1]
        print(f"codes={code_count:,} input_dim={input_dim} train/valid/test={len(split_data['train'].labels):,}/{len(split_data['valid'].labels):,}/{len(split_data['test'].labels):,}")

        code_tensor = torch.from_numpy(code_matrix).float().to(DEVICE)
        train_loader = make_loader(split_data["train"], shuffle=True)
        valid_loader = make_loader(split_data["valid"], shuffle=False)
        test_loader = make_loader(split_data["test"], shuffle=False)

        model = SiameseSpectralNet(input_dim, HIDDEN_DIM, EMBED_DIM, DROPOUT).to(DEVICE)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

        if USE_POS_WEIGHT:
            y_train = split_data["train"].labels
            pos = float((y_train == 1).sum())
            neg = float((y_train == 0).sum())
            pos_weight = torch.tensor([neg / max(1.0, pos)], device=DEVICE)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        else:
            criterion = nn.BCEWithLogitsLoss()

        scaler = make_scaler(USE_AMP)
        best_state = None
        best_valid_f1 = -1.0
        best_threshold = 0.5
        best_epoch = 0
        stale_epochs = 0
        history = []

        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            seen = 0
            pbar = tqdm(train_loader, desc=f"{graph_type.upper()} epoch {epoch:02d}", leave=False)
            for left_idx, right_idx, labels in pbar:
                left_idx = left_idx.to(DEVICE, non_blocking=True)
                right_idx = right_idx.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)
                with autocast_context(USE_AMP):
                    logits = model(code_tensor[left_idx], code_tensor[right_idx])
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                scaler.step(optimizer)
                scaler.update()
                batch_size = labels.numel()
                total_loss += float(loss.detach().cpu()) * batch_size
                seen += batch_size
                pbar.set_postfix(loss=total_loss / max(1, seen))

            train_loss = total_loss / max(1, seen)
            y_valid, valid_scores = predict_scores(model, code_tensor, valid_loader, f"{graph_type.upper()} valid")
            threshold = best_f1_threshold(y_valid, valid_scores)
            valid_metrics = metrics_at_threshold(y_valid, valid_scores, threshold)
            scheduler.step(valid_metrics["F1"])

            row = {
                "Method": graph_type.upper(),
                "epoch": epoch,
                "train_loss": train_loss,
                "valid_P": valid_metrics["P"],
                "valid_R": valid_metrics["R"],
                "valid_F1": valid_metrics["F1"],
                "valid_Acc": valid_metrics["Acc"],
                "threshold": threshold,
                "lr": optimizer.param_groups[0]["lr"],
            }
            history.append(row)
            print(f"epoch {epoch:02d} loss={train_loss:.4f} valid_f1={valid_metrics['F1']:.4f} valid_acc={valid_metrics['Acc']:.4f} thr={threshold:.4f}")

            if valid_metrics["F1"] > best_valid_f1:
                best_valid_f1 = valid_metrics["F1"]
                best_threshold = threshold
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                stale_epochs = 0
            else:
                stale_epochs += 1
                if stale_epochs >= PATIENCE:
                    print(f"Early stopping at epoch {epoch}; best epoch={best_epoch}, best_valid_f1={best_valid_f1:.4f}")
                    break

        if best_state is not None:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

        y_test, test_scores = predict_scores(model, code_tensor, test_loader, f"{graph_type.upper()} test")
        test_metrics = metrics_at_threshold(y_test, test_scores, best_threshold)
        record_language_breakdown(split_frames["test"], test_scores, best_threshold, dataset=DATASET_KEY_FOR_BREAKDOWN, method=f"{graph_type.upper()} + SNN", graph_type=graph_type)
        result = {
            "Method": graph_type.upper(),
            "BestEpoch": best_epoch,
            "BestValidF1": best_valid_f1,
            **test_metrics,
            "TrainPairs": int(len(split_data["train"].labels)),
            "TestPairs": int(len(split_data["test"].labels)),
            "TrainableParameters": int(sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)),
            "RuntimeSeconds": float(time.perf_counter() - graph_started),
            "RuntimeMinutes": float(time.perf_counter() - graph_started) / 60.0,
        }

        del model, optimizer, scheduler, code_tensor, train_loader, valid_loader, test_loader, vectors
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
        return result, history


    # =========================
    # Run all graph types
    # =========================
    all_results = []
    all_history = []

    for offset, graph_type in enumerate(GRAPH_TYPES):
        result, history = train_one_graph(graph_type, pairs_df, SEED + offset * 100)
        all_results.append(result)
        all_history.extend(history)
        pd.DataFrame(all_results).to_csv(RESULTS_PATH, index=False)
        pd.DataFrame(all_history).to_csv(HISTORY_PATH, index=False)
        print("Saved partial results:", RESULTS_PATH)

    results_df = pd.DataFrame(all_results)
    history_df = pd.DataFrame(all_history)

    print("\nFinal SNN results")
    display(results_df[["Method", "P", "R", "F1", "Acc", "BestEpoch", "BestValidF1", "Threshold", "TrainableParameters", "RuntimeSeconds", "RuntimeMinutes"]])
    print("Results:", RESULTS_PATH)
    print("History:", HISTORY_PATH)


    # =========================
    # Plot training curves
    # =========================
    if "history_df" not in globals() or history_df.empty:
        history_df = pd.read_csv(HISTORY_PATH)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4), dpi=140)
    for method, group in history_df.groupby("Method"):
        axes[0].plot(group["epoch"], group["train_loss"], marker="o", linewidth=1.5, label=method)
        axes[1].plot(group["epoch"], group["valid_F1"], marker="o", linewidth=1.5, label=method)
    axes[0].set_title("Train loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("BCE loss")
    axes[0].grid(alpha=0.25)
    axes[1].set_title("Validation F1")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("F1")
    axes[1].grid(alpha=0.25)
    axes[1].legend(loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(PLOT_PATH, bbox_inches="tight")
    print("Saved plot:", PLOT_PATH)
    plt.show()

    # Research-reproducibility manifest and enriched result table.
    # This is deliberately written after evaluation so measured runtime is final.
    import datetime as _datetime
    try:
        _gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
        _gpu_capability = ".".join(map(str, torch.cuda.get_device_capability(0))) if torch.cuda.is_available() else None
        _torch_version = torch.__version__
    except Exception:
        _gpu_name, _gpu_capability, _torch_version = "unavailable", None, "unavailable"
    _completed_utc = _datetime.datetime.now(_datetime.timezone.utc).isoformat()
    _run_seconds = float(time.perf_counter() - run_started)
    _shared_fields = {
        "Dataset": DATASET_KEY,
        "RunProfile": RUN_PROFILE,
        "Seed": int(SEED),
        "ConfiguredEpochs": int(EPOCHS) if "EPOCHS" in locals() else None,
        "BatchSize": int(BATCH_SIZE) if "BATCH_SIZE" in locals() else None,
        "LearningRate": float(LEARNING_RATE) if "LEARNING_RATE" in locals() else None,
        "WeightDecay": float(WEIGHT_DECAY) if "WEIGHT_DECAY" in locals() else None,
        "GPU": _gpu_name,
        "GPUCapability": _gpu_capability,
        "TorchVersion": _torch_version,
        "CompletedUTC": _completed_utc,
    }
    if "results_df" in locals():
        _result_table = results_df.copy()
    elif "result" in locals():
        _result_table = pd.DataFrame([result])
    elif "row" in locals():
        _result_table = pd.DataFrame([row])
    else:
        _result_table = pd.DataFrame()
    for _field, _value in _shared_fields.items():
        _result_table[_field] = _value
    if "RuntimeSeconds" not in _result_table.columns:
        _result_table["RuntimeSeconds"] = _run_seconds
    if "RuntimeMinutes" not in _result_table.columns:
        _result_table["RuntimeMinutes"] = _run_seconds / 60.0
    _result_path = RESULTS_PATH if "RESULTS_PATH" in locals() else out_path
    _result_table.to_csv(_result_path, index=False)
    _metadata = {
        **_shared_fields,
        "RunLabel": RUN_LABEL,
        "RuntimeSeconds": _run_seconds,
        "RuntimeMinutes": _run_seconds / 60.0,
        "RequestedPairCaps": {
            "train": MAX_TRAIN_PAIRS,
            "valid": MAX_VALID_PAIRS,
            "test": MAX_TEST_PAIRS,
        },
        "ModelConfiguration": {
            _name: locals().get(_name)
            for _name in (
                "MAX_AST_NODES", "MAX_AST_EDGES", "MAX_STATEMENTS", "MAX_NODE_TYPES", "MAX_NODES",
                "EMBED_DIM", "HIDDEN_DIM", "TREE_HIDDEN_DIM", "CODE_DIM", "DROPOUT",
                "GRAPH_TYPE", "GRAPH_TYPES", "K_EIGEN", "USE_EIGEN_STATS", "USE_GRAPH_STATS",
            ) if _name in locals()
        },
        "OutputFiles": {
            "results": str(_result_path),
            "history": str(HISTORY_PATH) if "HISTORY_PATH" in locals() else (str(history_path) if "history_path" in locals() else None),
        },
    }
    _metadata_path = WORK_DIR / f"{DATASET_KEY}_{RUN_LABEL}_run_metadata.json"
    _metadata_path.write_text(json.dumps(_metadata, indent=2, default=str), encoding="utf-8")
    print("Research metadata:", _metadata_path)
    if "results_df" in locals():
        results_df = _result_table

    if "results_df" in locals():
        return results_df.copy()
    if "result" in locals():
        return pd.DataFrame([result])
    if "row" in locals():
        return pd.DataFrame([row])
    raise RuntimeError("The baseline did not produce a result table.")


from IPython.display import display
import pandas as pd

all_dataset_results = {}
for current_dataset_key in DATASET_KEYS:
    print("\n" + "=" * 96)
    print(f"Running {current_dataset_key.upper()}")
    print("=" * 96)
    dataset_results = run_one_dataset(current_dataset_key)
    # The per-run result table already carries Dataset for research metadata.
    # Preserve a single authoritative value rather than inserting a duplicate column.
    if "Dataset" in dataset_results.columns:
        dataset_results["Dataset"] = current_dataset_key.upper()
    else:
        dataset_results.insert(0, "Dataset", current_dataset_key.upper())
    all_dataset_results[current_dataset_key] = dataset_results

print("\n" + "=" * 96)
print("Final result tables")
print("=" * 96)
for current_dataset_key in DATASET_KEYS:
    print(f"\n{current_dataset_key.upper()} results")
    display(all_dataset_results[current_dataset_key])

combined_results = pd.concat(
    [all_dataset_results[key] for key in DATASET_KEYS],
    ignore_index=True,
)
combined_path = Path("/kaggle/working") / f"{RUN_LABEL}_combined_dataset_results.csv"
combined_results.to_csv(combined_path, index=False)
print("Combined results:", combined_path)
